In [1]:
import sys
import socket

print(f"Node:{socket.gethostname()}")


Node:c027.curta.zedat.fu-berlin.de


In [2]:
import numpy as np
import rasterio
from rasterio.transform import from_bounds
from rasterio.crs import CRS
from scipy.stats import binned_statistic_2d
import matplotlib.pyplot as plt
import duckdb
from pathlib import Path

from photometry.models.hapke import HapkeModel
from photometry.core.types import GeometryBatch
from photometry.fitting.least_sq import LeastSquaresFitter
from photometry.aggregation.sphere_forward import sphere_forward_integrate

import spiceypy as spice
import plotly.graph_objects as go
from pathlib import Path

In [3]:
project_root = Path.cwd().resolve()
if not (project_root / "data").exists() and (project_root.parent / "data").exists():
    project_root = project_root.parent



parquet_path_survey_gaskell_dsk256_110825 = project_root / "data" / "geometry" / "gaskell_dsk256_110825" / "survey"/"*.parquet"

parquet_path_hamo_gaskell_dsk256_110825 = project_root / "data" / "geometry" / "gaskell_dsk256_110825" / "hamo"/"*.parquet"

parquet_path_lamo_gaskell_dsk256_110825 = project_root / "data" / "geometry" / "gaskell_dsk256_110825" / "lamo"/"*.parquet"

dsk_path = project_root/'data'/'spice_kernels'/'vesta_gaskell_256_110825.bds'

# Gaskell SPC 110825 DSK (NAIF archive)

In [ ]:
spice.kclear()
handle = spice.dasopr(str(dsk_path))
dladsc = spice.dlabfs(handle)

# Get number of vertices and plates
nv, np_plates = spice.dskz02(handle, dladsc)
vertices = spice.dskv02(handle, dladsc, 1, nv)
plates = spice.dskp02(handle, dladsc, 1, np_plates)

spice.dascls(handle)
spice.kclear()

# --- 2. Calculate Radial Elevation ---
# Calculate the exact distance from the center of Vesta for every vertex
radii = np.linalg.norm(vertices, axis=1)

# --- 3. Render with Elevation Color Coding ---
x, y, z = vertices.T
i, j, k = (plates - 1).T  

fig = go.Figure(data=[
    go.Mesh3d(
        x=x, y=y, z=z,
        i=i, j=j, k=k,
        intensity=radii,         # Color driven by actual distance from core
        colorscale='Turbo',      # 'Turbo' is excellent for deep topography contrast
        showscale=True,          # Turn on the legend
        colorbar=dict(
            title=dict(
                text="Radius (km)",
                font=dict(color='white', size=14)
            ),
            tickfont=dict(color='white', size=12),
            xpad=20,
            thickness=15
        ),
        flatshading=True,        # CRITICAL: Keeps the faceted "plate" look visible
        lighting=dict(
            ambient=0.3, 
            diffuse=0.9, 
            roughness=0.5, 
            specular=0.1         # Keep specular low so it looks like rock, not plastic
        ),
        lightposition=dict(x=1000, y=1000, z=500)
    )
])

fig.update_layout(
    title=dict(
        text="Vesta DSK256: Radial Elevation Map",
        font=dict(color='white', size=18)
    ),
    scene=dict(
        aspectmode='data',       # Prevents mathematical stretching
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        zaxis=dict(visible=False)
    ),
    paper_bgcolor='black',       # Perfect for presentation slides
    margin=dict(l=0, r=0, b=0, t=50)
)

fig.show()

ValueError: Invalid property specified for object of type plotly.graph_objs.mesh3d.ColorBar: 'titlefont'

Did you mean "tickfont"?

    Valid properties:
        bgcolor
            Sets the color of padded area.
        bordercolor
            Sets the axis line color.
        borderwidth
            Sets the width (in px) or the border enclosing this
            color bar.
        dtick
            Sets the step in-between ticks on this axis. Use with
            `tick0`. Must be a positive number, or special strings
            available to "log" and "date" axes. If the axis `type`
            is "log", then ticks are set every 10^(n*dtick) where n
            is the tick number. For example, to set a tick mark at
            1, 10, 100, 1000, ... set dtick to 1. To set tick marks
            at 1, 100, 10000, ... set dtick to 2. To set tick marks
            at 1, 5, 25, 125, 625, 3125, ... set dtick to
            log_10(5), or 0.69897000433. "log" has several special
            

In [ ]:
image_candidates = duckdb.sql(f"""
    SELECT 
        image_id,
        AVG(incidence)   AS mean_inc,
        AVG(emission)    AS mean_emi,
        AVG(phase)       AS mean_phase,
        STDDEV_SAMP(iof) AS std_iof,
        AVG(iof)         AS mean_iof,
        COUNT(*)         AS n_pixels
    FROM read_parquet('{parquet_path_survey_gaskell_dsk256_110825}')
                              
    WHERE image_id LIKE '%F1%'
      AND incidence < 45.0
      AND emission  < 30.0
      AND iof > 0.01
    GROUP BY image_id
    HAVING COUNT(*) > 500000
    ORDER BY STDDEV_SAMP(iof) DESC
    LIMIT 20
""").df()

image_candidates


In [4]:
print(f"Rows returned: {len(image_candidates)}")
# image_candidates

NameError: name 'image_candidates' is not defined

In [97]:
image_id = image_candidates.iloc[10]["image_id"]
image_id

'FC21B0006719_11241181536F1C'

In [ ]:
query = f"""
    SELECT pixel_x, pixel_y, iof, incidence
    FROM read_parquet('{parquet_path_survey_gaskell_dsk256_110825}')
    WHERE image_id = '{image_id}'
      AND iof > 0.0156 * COS(RADIANS(incidence))
"""
df = duckdb.sql(query).df()
print(f"{image_id}: {len(df):,} pixels loaded")

# 2. Reconstruct the 1024x1024 raw image grid
raw_img = np.full((1024, 1024), np.nan)
x = df['pixel_x'].astype(int).to_numpy()
y = df['pixel_y'].astype(int).to_numpy()
raw_img[y, x] = df['iof'].to_numpy()

# Calculate dynamic display range
vmin, vmax = np.nanpercentile(raw_img[raw_img > 0.01], [2, 98])

# 3. Plot single-panel figure
fig, ax = plt.subplots(figsize=(6, 5.5), dpi=300)

im = ax.imshow(raw_img, cmap='gray', vmin=vmin, vmax=vmax, origin='upper')
ax.set_title(f'Raw Calibrated I/F — {image_id}', fontsize=12, fontweight='bold')
ax.axis('off')

cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label('Reflectance (I/F)', fontsize=10)
cbar.ax.tick_params(labelsize=9)

plt.tight_layout()
plt.show()

In [ ]:
model = HapkeModel(
    enable_shoe=True, enable_roughness=True,
    fixed_parameters={'B0': 1.03, 'h': 0.04}
)
model.parameters.update({'w': 0.4617, 'g': -0.3323, 'theta_bar': 7.61})



query = f"""
    SELECT pixel_x, pixel_y, iof, incidence, emission, phase
    FROM read_parquet('{parquet_path_survey_gaskell_dsk256_110825}')
    WHERE image_id = '{image_id}'
      AND iof > 0.0156 * COS(RADIANS(incidence))
"""
df = duckdb.sql(query).df()
print(f"{image_id}: {len(df):,} pixels loaded")

geom = GeometryBatch(
    incidence=np.deg2rad(df['incidence'].to_numpy()),
    emission=np.deg2rad(df['emission'].to_numpy()),
    phase=np.deg2rad(df['phase'].to_numpy())
)
geom_std = GeometryBatch(
    incidence=np.deg2rad(np.full(len(df), 30.0)),
    emission=np.deg2rad(np.zeros(len(df))),
    phase=np.deg2rad(np.full(len(df), 30.0))
)
predicted_real     = model._reflectance_numpy(geom)
predicted_standard = model._reflectance_numpy(geom_std)

correction_factor = np.where(
    predicted_real > 0.005,
    np.clip(predicted_standard / np.maximum(predicted_real, 1e-6), 0.5, 1.8),
    np.nan
)
df['iof_corrected'] = df['iof'].to_numpy() * correction_factor

raw_img       = np.full((1024, 1024), np.nan)
corrected_img = np.full((1024, 1024), np.nan)
x = df['pixel_x'].astype(int).to_numpy()
y = df['pixel_y'].astype(int).to_numpy()
raw_img[y, x]       = df['iof'].to_numpy()
corrected_img[y, x] = df['iof_corrected'].to_numpy()

ratio_img = corrected_img / raw_img

# ratio_img = np.fliplr(ratio_img)  # == correction_factor, spatially reconstructed

vmin, vmax = np.nanpercentile(raw_img[raw_img > 0.01], [2, 98])
rmin, rmax = np.nanpercentile(ratio_img[~np.isnan(ratio_img)], [2, 98])

# ---- Publication-style 3-panel figure, white background ----
fig, axes = plt.subplots(1, 3, figsize=(16, 5.5), dpi=300)

im0 = axes[0].imshow(raw_img, cmap='gray', vmin=vmin, vmax=vmax, origin='upper')
axes[0].set_title('(a) Raw calibrated I/F', fontsize=11)
axes[0].axis('off')

im1 = axes[1].imshow(corrected_img, cmap='gray', vmin=vmin, vmax=vmax, origin='upper')
axes[1].set_title('(b) Photometrically corrected I/F\n' + r'($i$=30°, $e$=0°, $\alpha$=30°)', fontsize=11)
axes[1].axis('off')

im2 = axes[2].imshow(ratio_img, cmap='RdBu_r', vmin=rmin, vmax=rmax, origin='upper')
axes[2].set_title('(c) Correction factor\n(corrected / raw)', fontsize=11)
axes[2].axis('off')

cbar0 = fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)
cbar0.set_label('Reflectance (I/F)', fontsize=9)
cbar0.ax.tick_params(labelsize=8)

cbar1 = fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)
cbar1.set_label('Reflectance (I/F)', fontsize=9)
cbar1.ax.tick_params(labelsize=8)

cbar2 = fig.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)
cbar2.set_label('Correction factor', fontsize=9)
cbar2.ax.tick_params(labelsize=8)
fig.suptitle(f'Photometric Correction — {image_id}\n'
             r'Hapke: $w$=0.4617, $g$=−0.3323, $\bar\theta$=7.61°',
             fontsize=12, y=1.02)

plt.tight_layout()

plt.show()

print(f"Correction factor range shown: {rmin:.3f} to {rmax:.3f}")

NameError: name 'image_id' is not defined

In [87]:
print(f"Mean incidence: {df['incidence'].mean():.1f}°")
print(f"Mean emission: {df['emission'].mean():.1f}°")
print(f"Mean phase: {df['phase'].mean():.1f}°")


print(f"Reference geometry: i=30°, e=0°, α=30°")
print(f"This image's actual mean geometry: i={df['incidence'].mean():.1f}°, "
      f"e={df['emission'].mean():.1f}°, α={df['phase'].mean():.1f}°")
print(f"Correction factor stats: min={np.nanmin(ratio_img):.3f}, "
      f"max={np.nanmax(ratio_img):.3f}, median={np.nanmedian(ratio_img):.3f}")

Mean incidence: 29.1°
Mean emission: 27.6°
Mean phase: 17.9°
Reference geometry: i=30°, e=0°, α=30°
This image's actual mean geometry: i=29.1°, e=27.6°, α=17.9°
Correction factor stats: min=0.500, max=1.800, median=0.774


In [88]:
high_correction_mask = ratio_img > 1.5
local_inc = raw_img.copy()  # placeholder — need incidence reconstructed the same way as raw_img/corrected_img
# reconstruct incidence_img the same way you built raw_img, then:'
print(f"Mean local incidence where correction>1.5: check against 50° threshold")

Mean local incidence where correction>1.5: check against 50° threshold


In [89]:
incidence_img = np.full((1024, 1024), np.nan)
incidence_img[y, x] = df['incidence'].to_numpy()

high_corr_mask = (ratio_img > 1.4) | (ratio_img < 0.7)
print(f"Mean local incidence in strong-correction pixels (|factor-1|>0.4-ish): "
      f"{np.nanmean(incidence_img[high_corr_mask]):.1f}°")
print(f"Max local incidence in strong-correction pixels: "
      f"{np.nanmax(incidence_img[high_corr_mask]):.1f}°")


Mean local incidence in strong-correction pixels (|factor-1|>0.4-ish): 53.8°
Max local incidence in strong-correction pixels: 89.4°
